In [ ]:

import pandas as pd
from pathlib import Path


# ============================================================
# STEP 13 — BUILD MASTER ATTENDANCE DAILY
# ============================================================

# ============================================================
# 1. CONFIGURATION
# ============================================================

ATTENDANCE_FILE = Path(
    "output/second_reports/attendance_feature_engineered.xlsx"
)

CALENDAR_FILE = Path(
    "output/dim_calendar.csv"
)

OUTPUT_DIR = Path("output/master")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "master_attendance_daily.csv"
QUALITY_REPORT_FILE = (
    OUTPUT_DIR / "master_attendance_daily_quality_report.csv"
)

# Excel sheet
ATTENDANCE_SHEET = 0

# Employee identification
EMPLOYEE_KEY = "employee_id"
DATE_KEY = "attendance_date"

# HO identification
HO_COST_CENTER_PREFIX = "9"

# Required attendance columns
REQUIRED_ATTENDANCE_COLUMNS = [
    "employee_id",
    "employee_name",
    "attendance_date",
    "cost_center_code",
    "cost_center_type",
    "weekly_work_quota",
    "has_attendance_activity",
]

# Required calendar columns
REQUIRED_CALENDAR_COLUMNS = [
    "attendance_date",
    "day_of_week",
    "is_national_holiday",
]


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def convert_boolean(series):
    """
    Convert common boolean representations into pandas
    nullable boolean values.
    """

    mapping = {
        "TRUE": True,
        "FALSE": False,
        "1": True,
        "0": False,
        "YES": True,
        "NO": False,
    }

    if pd.api.types.is_bool_dtype(series):
        return series.astype("boolean")

    normalized = (
        series.astype("string")
        .str.strip()
        .str.upper()
    )

    return normalized.map(mapping).astype("boolean")


def normalize_date(series):
    """
    Normalize date values without changing the actual date.
    """

    return pd.to_datetime(
        series,
        errors="coerce"
    ).dt.normalize()


def add_quality_result(results, metric, value, status, note=""):
    results.append({
        "metric": metric,
        "value": value,
        "status": status,
        "note": note,
    })


# ============================================================
# 3. CHECK INPUT FILES
# ============================================================

if not ATTENDANCE_FILE.exists():
    raise FileNotFoundError(
        f"Attendance file not found: {ATTENDANCE_FILE}"
    )

if not CALENDAR_FILE.exists():
    raise FileNotFoundError(
        f"Calendar file not found: {CALENDAR_FILE}"
    )

print("=" * 60)
print("STEP 13: BUILD MASTER ATTENDANCE DAILY")
print("=" * 60)


# ============================================================
# 4. LOAD ATTENDANCE DATA (.XLSX)
# ============================================================

print("\n[1/9] Loading attendance data...")

attendance = pd.read_excel(
    ATTENDANCE_FILE,
    sheet_name=ATTENDANCE_SHEET,
    dtype={
        "employee_id": "string",
        "cost_center_code": "string",
    }
)

attendance.columns = (
    attendance.columns
    .str.strip()
)

missing_attendance_columns = [
    col for col in REQUIRED_ATTENDANCE_COLUMNS
    if col not in attendance.columns
]

if missing_attendance_columns:
    raise ValueError(
        "Missing attendance columns: "
        f"{missing_attendance_columns}"
    )

attendance[EMPLOYEE_KEY] = (
    attendance[EMPLOYEE_KEY]
    .astype("string")
    .str.strip()
)

attendance[DATE_KEY] = normalize_date(
    attendance[DATE_KEY]
)

if attendance[EMPLOYEE_KEY].isna().any():
    raise ValueError(
        "Attendance contains missing employee_id."
    )

if attendance[DATE_KEY].isna().any():
    raise ValueError(
        "Attendance contains invalid attendance_date."
    )

original_attendance_rows = len(attendance)

print(f"Attendance rows : {original_attendance_rows:,}")
print(f"Attendance cols : {len(attendance.columns):,}")


# ============================================================
# 5. LOAD CALENDAR DATA (.CSV)
# ============================================================

print("\n[2/9] Loading calendar data...")

calendar = pd.read_csv(
    CALENDAR_FILE,
    dtype={"attendance_date": "string"},
    low_memory=False
)

calendar.columns = (
    calendar.columns
    .str.strip()
)

missing_calendar_columns = [
    col for col in REQUIRED_CALENDAR_COLUMNS
    if col not in calendar.columns
]

if missing_calendar_columns:
    raise ValueError(
        "Missing calendar columns: "
        f"{missing_calendar_columns}"
    )

calendar[DATE_KEY] = normalize_date(
    calendar[DATE_KEY]
)

if calendar[DATE_KEY].isna().any():
    raise ValueError(
        "Calendar contains invalid attendance_date."
    )

if calendar[DATE_KEY].duplicated().any():
    duplicated_dates = (
        calendar.loc[
            calendar[DATE_KEY].duplicated(keep=False),
            DATE_KEY
        ]
        .drop_duplicates()
        .head(10)
        .tolist()
    )

    raise ValueError(
        "Calendar contains duplicate dates: "
        f"{duplicated_dates}"
    )

calendar_rows = len(calendar)

print(f"Calendar rows : {calendar_rows:,}")
print(f"Calendar cols : {len(calendar.columns):,}")


# ============================================================
# 6. PREPARE CALENDAR DATA
# ============================================================

print("\n[3/9] Preparing calendar attributes...")

calendar["day_of_week"] = pd.to_numeric(
    calendar["day_of_week"],
    errors="coerce"
)

calendar["is_national_holiday"] = (
    convert_boolean(
        calendar["is_national_holiday"]
    )
)

if calendar["day_of_week"].isna().any():
    raise ValueError(
        "Calendar contains invalid day_of_week."
    )

if calendar["is_national_holiday"].isna().any():
    raise ValueError(
        "Calendar contains missing is_national_holiday."
    )

# Monday = 0, Tuesday = 1, ..., Sunday = 6

calendar["is_weekday_calendar"] = (
    calendar["day_of_week"].between(0, 4)
)

# HO works Monday-Friday, excluding national holidays.
# Joint leave is not automatically treated as a national holiday.

calendar["is_ho_workday"] = (
    calendar["is_weekday_calendar"]
    & ~calendar["is_national_holiday"].astype(bool)
)

print("Calendar attributes prepared.")


# ============================================================
# 7. JOIN ATTENDANCE WITH CALENDAR
# ============================================================

print("\n[4/9] Joining attendance with calendar...")

master = attendance.merge(
    calendar,
    on=DATE_KEY,
    how="left",
    validate="many_to_one",
    indicator="_calendar_match",
    suffixes=("", "_calendar")
)

unmatched_dates = master[
    master["_calendar_match"] != "both"
]

if not unmatched_dates.empty:
    missing_dates = (
        unmatched_dates[DATE_KEY]
        .drop_duplicates()
        .head(10)
        .tolist()
    )

    raise ValueError(
        "Some attendance dates do not exist in calendar. "
        f"Examples: {missing_dates}"
    )

master = master.drop(
    columns=["_calendar_match"]
)

if len(master) != original_attendance_rows:
    raise ValueError(
        "Attendance row count changed after calendar JOIN."
    )

print("Calendar JOIN successful.")
print(f"Joined rows: {len(master):,}")


# ============================================================
# 8. GENERATE DAILY ATTENDANCE BUSINESS RULES
# ============================================================

print("\n[5/9] Generating daily business rules...")

# ------------------------------------------------------------
# 8.1 Normalize employee classification
# ------------------------------------------------------------

master["cost_center_type"] = (
    master["cost_center_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

master["cost_center_code"] = (
    master["cost_center_code"]
    .astype("string")
    .str.strip()
)

code_is_ho = (
    master["cost_center_code"]
    .str.startswith(
        HO_COST_CENTER_PREFIX,
        na=False
    )
)

type_is_ho = (
    master["cost_center_type"] == "FIXED_5"
)

type_is_non_ho = (
    master["cost_center_type"] == "NON_FIXED_6"
)

# Use the existing classification as primary.
# Cost center prefix is used as fallback when type is missing.

master["is_ho_employee"] = (
    type_is_ho
    | (
        master["cost_center_type"].isna()
        & code_is_ho
    )
).astype("boolean")

master["daily_attendance_policy"] = (
    master["is_ho_employee"]
    .map({
        True: "FIXED_DAILY",
        False: "WEEKLY_QUOTA",
    })
    .astype("string")
)

# ------------------------------------------------------------
# 8.2 Normalize attendance activity
# ------------------------------------------------------------

master["has_attendance_activity"] = (
    convert_boolean(
        master["has_attendance_activity"]
    )
)

if master["has_attendance_activity"].isna().any():
    raise ValueError(
        "Missing has_attendance_activity values."
    )

# ------------------------------------------------------------
# 8.3 Expected workday
# ------------------------------------------------------------

# HO:
# TRUE  = Monday-Friday, excluding national holidays
# FALSE = Weekend or national holiday
#
# Non-HO:
# UNKNOWN because daily roster is unavailable.

master["is_expected_workday"] = (
    master["is_ho_employee"]
    .map({True: pd.NA, False: pd.NA})
    .astype("boolean")
)

ho_mask = master["is_ho_employee"].fillna(False)

master.loc[
    ho_mask,
    "is_expected_workday"
] = master.loc[
    ho_mask,
    "is_ho_workday"
].astype("boolean")

# ------------------------------------------------------------
# 8.4 Daily compliance applicability
# ------------------------------------------------------------

master["is_daily_compliance_applicable"] = (
    ho_mask
    & master["is_expected_workday"].fillna(False)
).astype("boolean")

# ------------------------------------------------------------
# 8.5 Daily attendance compliance
# ------------------------------------------------------------

master["is_daily_attendance_compliant"] = (
    pd.Series(
        pd.NA,
        index=master.index,
        dtype="boolean"
    )
)

applicable_mask = (
    master["is_daily_compliance_applicable"]
    .fillna(False)
)

master.loc[
    applicable_mask,
    "is_daily_attendance_compliant"
] = master.loc[
    applicable_mask,
    "has_attendance_activity"
].astype("boolean")

# ------------------------------------------------------------
# 8.6 Daily absence candidate
# ------------------------------------------------------------

master["is_daily_absence_candidate"] = (
    pd.Series(
        pd.NA,
        index=master.index,
        dtype="boolean"
    )
)

master.loc[
    applicable_mask,
    "is_daily_absence_candidate"
] = (
    ~master.loc[
        applicable_mask,
        "has_attendance_activity"
    ].astype(bool)
)

# ------------------------------------------------------------
# 8.7 Daily attendance status
# ------------------------------------------------------------

master["daily_attendance_status"] = "UNKNOWN"

# HO expected workday + activity
master.loc[
    applicable_mask
    & master["has_attendance_activity"].fillna(False),
    "daily_attendance_status"
] = "PRESENT"

# HO expected workday + no activity
master.loc[
    applicable_mask
    & ~master["has_attendance_activity"].fillna(False),
    "daily_attendance_status"
] = "NO_ACTIVITY"

# HO holiday + activity
ho_holiday_activity = (
    ho_mask
    & master["is_national_holiday"].fillna(False)
    & master["has_attendance_activity"].fillna(False)
)

master.loc[
    ho_holiday_activity,
    "daily_attendance_status"
] = "HOLIDAY_WORK_ACTIVITY"

# HO holiday + no activity
ho_holiday_no_activity = (
    ho_mask
    & master["is_national_holiday"].fillna(False)
    & ~master["has_attendance_activity"].fillna(False)
)

master.loc[
    ho_holiday_no_activity,
    "daily_attendance_status"
] = "HOLIDAY"

# HO weekend + activity
ho_weekend_activity = (
    ho_mask
    & master["is_weekend"].fillna(False)
    & master["has_attendance_activity"].fillna(False)
)

master.loc[
    ho_weekend_activity,
    "daily_attendance_status"
] = "NON_WORKDAY_ACTIVITY"

# HO weekend + no activity
ho_weekend_no_activity = (
    ho_mask
    & master["is_weekend"].fillna(False)
    & ~master["has_attendance_activity"].fillna(False)
)

master.loc[
    ho_weekend_no_activity,
    "daily_attendance_status"
] = "NON_WORKDAY"

# Non-HO with activity
non_ho_activity = (
    ~ho_mask
    & master["has_attendance_activity"].fillna(False)
)

master.loc[
    non_ho_activity,
    "daily_attendance_status"
] = "ATTENDANCE_ACTIVITY"

# Non-HO without activity, if such a row exists
non_ho_no_activity = (
    ~ho_mask
    & ~master["has_attendance_activity"].fillna(False)
)

master.loc[
    non_ho_no_activity,
    "daily_attendance_status"
] = "NO_ACTIVITY_RECORDED"

# ------------------------------------------------------------
# 8.8 Classification consistency check
# ------------------------------------------------------------

master["cost_center_classification_conflict"] = (
    code_is_ho != type_is_ho
) & master["cost_center_type"].notna()


# ============================================================
# 9. DATA QUALITY VALIDATION
# ============================================================

print("\n[6/9] Running quality validation...")

quality_results = []

# ------------------------------------------------------------
# Row preservation
# ------------------------------------------------------------

add_quality_result(
    quality_results,
    "attendance_rows_before_join",
    original_attendance_rows,
    "INFO"
)

add_quality_result(
    quality_results,
    "attendance_rows_after_join",
    len(master),
    "PASS" if len(master) == original_attendance_rows else "FAIL"
)

# ------------------------------------------------------------
# Employee preservation
# ------------------------------------------------------------

employees_before = attendance[
    EMPLOYEE_KEY
].nunique()

employees_after = master[
    EMPLOYEE_KEY
].nunique()

add_quality_result(
    quality_results,
    "unique_employees_before_join",
    employees_before,
    "INFO"
)

add_quality_result(
    quality_results,
    "unique_employees_after_join",
    employees_after,
    "PASS" if employees_before == employees_after else "FAIL"
)

# ------------------------------------------------------------
# Calendar matching
# ------------------------------------------------------------

add_quality_result(
    quality_results,
    "unmatched_calendar_dates",
    0,
    "PASS"
)

# ------------------------------------------------------------
# Duplicate employee-date detection
# ------------------------------------------------------------

duplicate_mask = master.duplicated(
    subset=[EMPLOYEE_KEY, DATE_KEY],
    keep=False
)

duplicate_rows = int(duplicate_mask.sum())

master["is_duplicate_employee_date"] = (
    duplicate_mask
)

add_quality_result(
    quality_results,
    "duplicate_employee_date_rows",
    duplicate_rows,
    "WARNING" if duplicate_rows > 0 else "PASS",
    "Rows are preserved. No automatic deletion performed."
)

# ------------------------------------------------------------
# Employee classification
# ------------------------------------------------------------

add_quality_result(
    quality_results,
    "HO_employee_rows",
    int(ho_mask.sum()),
    "INFO"
)

add_quality_result(
    quality_results,
    "non_HO_employee_rows",
    int((~ho_mask).sum()),
    "INFO"
)

classification_conflicts = int(
    master["cost_center_classification_conflict"].sum()
)

add_quality_result(
    quality_results,
    "cost_center_classification_conflicts",
    classification_conflicts,
    "WARNING" if classification_conflicts > 0 else "PASS",
    "Compare cost_center_code prefix with cost_center_type."
)

# ------------------------------------------------------------
# Non-HO daily compliance validation
# ------------------------------------------------------------

non_ho_mask = ~ho_mask

non_ho_daily_compliance = (
    master.loc[
        non_ho_mask,
        "is_daily_compliance_applicable"
    ]
    .fillna(False)
    .sum()
)

add_quality_result(
    quality_results,
    "non_HO_daily_compliance_applicable_rows",
    int(non_ho_daily_compliance),
    "PASS" if non_ho_daily_compliance == 0 else "FAIL",
    "Non-HO must not be evaluated using daily compliance."
)

# ------------------------------------------------------------
# Save quality report
# ------------------------------------------------------------

quality_report = pd.DataFrame(
    quality_results
)

quality_report.to_csv(
    QUALITY_REPORT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 10. FINAL COLUMN ORGANIZATION
# ============================================================

print("\n[7/9] Organizing final columns...")

# Keep all original attendance columns.
# Keep all calendar columns.
# Keep all newly generated business rule columns.

business_rule_columns = [
    "is_ho_employee",
    "daily_attendance_policy",
    "is_expected_workday",
    "is_daily_compliance_applicable",
    "is_daily_attendance_compliant",
    "is_daily_absence_candidate",
    "daily_attendance_status",
    "cost_center_classification_conflict",
    "is_duplicate_employee_date",
]

final_columns = list(attendance.columns)

for col in calendar.columns:
    if col not in final_columns:
        final_columns.append(col)

for col in business_rule_columns:
    if col not in final_columns:
        final_columns.append(col)

master = master[final_columns]


# ============================================================
# 11. EXPORT MASTER DAILY
# ============================================================

print("\n[8/9] Exporting master attendance daily...")

master.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 12. FINAL SUMMARY
# ============================================================

print("\n[9/9] Process completed.")

print("=" * 60)
print("MASTER ATTENDANCE DAILY SUMMARY")
print("=" * 60)

print(f"Input attendance rows : {original_attendance_rows:,}")
print(f"Input calendar rows   : {calendar_rows:,}")
print(f"Output master rows    : {len(master):,}")
print(f"Output columns        : {len(master.columns):,}")

print(
    f"Unique employees      : "
    f"{master[EMPLOYEE_KEY].nunique():,}"
)

print(
    f"HO records            : "
    f"{int(ho_mask.sum()):,}"
)

print(
    f"Non-HO records        : "
    f"{int(non_ho_mask.sum()):,}"
)

print(
    f"Duplicate employee-date rows: "
    f"{duplicate_rows:,}"
)

print("\nDaily attendance status:")
print(
    master["daily_attendance_status"]
    .value_counts(dropna=False)
)

print("\nOutput files:")
print(f"Master  : {OUTPUT_FILE}")
print(f"Quality : {QUALITY_REPORT_FILE}")
print("\nSTEP 13 COMPLETED SUCCESSFULLY.")


STEP 13: BUILD MASTER ATTENDANCE DAILY

[1/9] Loading attendance data...
Attendance rows : 27,711
Attendance cols : 70

[2/9] Loading calendar data...
Calendar rows : 371
Calendar cols : 22

[3/9] Preparing calendar attributes...
Calendar attributes prepared.

[4/9] Joining attendance with calendar...
Calendar JOIN successful.
Joined rows: 27,711

[5/9] Generating daily business rules...

[6/9] Running quality validation...

[7/9] Organizing final columns...

[8/9] Exporting master attendance daily...

[9/9] Process completed.
MASTER ATTENDANCE DAILY SUMMARY
Input attendance rows : 27,711
Input calendar rows   : 371
Output master rows    : 27,711
Output columns        : 102
Unique employees      : 5,729
HO records            : 2,106
Non-HO records        : 25,605
Duplicate employee-date rows: 0

Daily attendance status:
daily_attendance_status
ATTENDANCE_ACTIVITY      25605
PRESENT                   2005
NON_WORKDAY_ACTIVITY        73
HOLIDAY_WORK_ACTIVITY       28
Name: count, dtype: 